# Amma Sewana — Evaluation: Base vs Fine-tuned Gemma 4 E2B

**No unsloth required.** Uses plain `transformers` + `peft` for inference.

**Before running:** GPU T4 x2 | Internet ON | `HF_TOKEN` secret attached | Factory Reset kernel first.

Run top to bottom once.

## 1. Install

In [ ]:
%%capture
!pip install -q --upgrade transformers peft accelerate bitsandbytes huggingface_hub

In [ ]:
import torch, re, textwrap, pandas as pd
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
print("All imports OK")

## 2. HuggingFace login

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=token)
print("HF login OK")

## 3. Eval questions, scoring, inference helper

In [ ]:
SYSTEM_INSTRUCTION = (
    "You are Amma Sewana, an AI assistant for Sri Lanka Public Health Midwives (PHMs). "
    "Help PHMs assess pregnant mothers and answer clinical questions. "
    "Follow Sri Lanka Ministry of Health maternal care guidelines, NICE and RCOG guidelines. "
    "Always state risk level (LOW/MEDIUM/HIGH/URGENT) and immediate action where relevant. "
    "Reply in the same language as the question (Sinhala, Tamil, or English)."
)

EVAL_QUESTIONS = [
    {"id":"EN-01","lang":"en","topic":"hypertension",
     "question":"A 38-year-old primigravida at 36 weeks has BP 150/98, 1+ proteinuria, mild hand swelling. No headache or visual symptoms. Risk level and next step?"},
    {"id":"EN-02","lang":"en","topic":"hypertension",
     "question":"A woman at 26 weeks with chronic hypertension on methyldopa 250mg TID presents with BP 168/112 despite medication. What should I do?"},
    {"id":"EN-03","lang":"en","topic":"eclampsia",
     "question":"A woman at 30 weeks has a generalised tonic-clonic seizure in the clinic. No prior epilepsy. Immediate steps?"},
    {"id":"EN-04","lang":"en","topic":"reduced_fetal_movement",
     "question":"A mother at 32 weeks has not felt the baby move since yesterday morning. Good movements before. Recommended assessment?"},
    {"id":"EN-05","lang":"en","topic":"reduced_fetal_movement",
     "question":"Should pregnant women use kick charts? What does current RCOG guidance say about routine kick counting?"},
    {"id":"EN-06","lang":"en","topic":"postpartum_haemorrhage",
     "question":"A woman delivers at a peripheral clinic with 800ml blood loss within 30 min of birth. Placenta delivered. Immediate management?"},
    {"id":"EN-07","lang":"en","topic":"postpartum_haemorrhage",
     "question":"What uterotonic drugs are used for prevention and treatment of PPH, and what are the preferred first-line choices?"},
    {"id":"EN-08","lang":"en","topic":"anaemia",
     "question":"A pregnant woman at 28 weeks has Hb 9.2 g/dL. She has been on oral iron for 4 weeks with no improvement. Next step?"},
    {"id":"EN-09","lang":"en","topic":"anaemia",
     "question":"At what haemoglobin level should a pregnant woman be referred for blood transfusion?"},
    {"id":"EN-10","lang":"en","topic":"diabetes",
     "question":"A woman at 26 weeks has a random blood glucose of 11.2 mmol/L. No prior diabetes. What investigations are required?"},
    {"id":"EN-11","lang":"en","topic":"diabetes",
     "question":"Target fasting and 1-hour post-meal blood glucose for a pregnant woman with gestational diabetes on diet control?"},
    {"id":"EN-12","lang":"en","topic":"antenatal_care",
     "question":"A woman at 10 weeks, first visit, BMI 36, no known conditions. Complete booking assessment the PHM should perform?"},
    {"id":"SI-01","lang":"si","topic":"hypertension",
     "question":"ගර්භිනී කාන්තාවක්ගේ රුධිර පීඩනය 155/100 mmHg. කිසිදු රෝග ලක්ෂණයක් නැත. ජෝජු ක්‍රියාමාරගය කුමක්ද?"},
    {"id":"SI-02","lang":"si","topic":"hypertension",
     "question":"ගර්භිනීව සිටින කාන්තාවක් රුධිර පීඩනය 170/114, ප්‍රෝටීනිහිතය 3+, හිසරදය සහ ද්‍රුෂ්ටි ගැටලු. හදිස්සි ක්‍රියාව කුමක්ද?"},
    {"id":"SI-03","lang":"si","topic":"eclampsia",
     "question":"සතිය 38 ගර්භිනී කාන්තාවකට සෙලවීම් ගිහිල්ලා දැන් හිත නොතේරේ. මේ අවස්ථාවේ ක්‍රියාවලිය?"},
    {"id":"SI-04","lang":"si","topic":"reduced_fetal_movement",
     "question":"සතිය 36 ගර්භිනී කාන්තාවකට ඊයේ ඉදලා දරුවාගේ සෙලවීම් 4ක් විතරක් දැනුණා. PHM ලෙස ඔබ කුමක් කරනවාද?"},
    {"id":"SI-05","lang":"si","topic":"reduced_fetal_movement",
     "question":"ගර්භිනී කාන්තාවකට CTG කළ යුතු නම් කොහේ යොමු කළ යුතුද?"},
    {"id":"SI-06","lang":"si","topic":"postpartum_haemorrhage",
     "question":"ප්‍රසූතිය සිදු වී මිනිත්තු 15 ගත්, ජෙරනිය බිහිවී නැත, රුධිරය ගලා යයි. PHM ලෙස කළ යුත්තේ කුමක්ද?"},
    {"id":"SI-07","lang":"si","topic":"anaemia",
     "question":"ගර්භිනී කාන්තාවකගේ හිමොග්ලොබින් 8.5 g/dL, සතිය 30, හුස්ම ගන්නට අමාරු. කළ යුත්තේ කුමක්ද?"},
    {"id":"SI-08","lang":"si","topic":"anaemia",
     "question":"ගර්භිනී කාන්තාවකට යකඩ ටැබ්ලට් දෙන විට ශරීරය ජොරපිට ඇතිවේ. වෙනත් ක්‍රමයකින් ලබාදිය හැකිද?"},
    {"id":"SI-09","lang":"si","topic":"diabetes",
     "question":"ගර්භිනී කාන්තාවකගේ රුධිර ග්ලූකෝස් 6.8 mmol/L. GDM ඇතිදැයි නිශ්චිතව දැනගන්නේ කෙසේද?"},
    {"id":"SI-10","lang":"si","topic":"diabetes",
     "question":"GDM ඇති ගර්භිනී කාන්තාවකට ඉන්සියුලින් ලබා දිනා විට කකුල් ලෙලදෙයි, දහඩිය, කතා කරන්නේ නෑ. PHM ලෙස කළ යුත්තේ කුමක්ද?"},
    {"id":"SI-11","lang":"si","topic":"antenatal_care",
     "question":"ගර්භ සතිය 12 දී PHM සිදු කළ යුතු ඇගයීම් මොනවාද?"},
    {"id":"SI-12","lang":"si","topic":"antenatal_care",
     "question":"ගර්භිනී කාන්තාවක් දුම්කොළ පානය කරනවා. ගර්භනී සමයේ දුම්කොළ ගැන මගපෙන්වීම කුමක්ද?"}
]

SINHALA_RE = re.compile(r'[඀-෿]')

def score_response(q, answer):
    up = answer.upper()
    r = int(bool(re.search(r'\b(LOW|MEDIUM|HIGH|URGENT)\b', up)))
    s = int(bool(re.search(r'\b(MOH|NICE|RCOG|NG133|NG3|NG201|GTG)\b', up)))
    a = int(len(answer.strip()) >= 60)
    l = int(bool(SINHALA_RE.search(answer))) if q['lang'] == 'si' else 1
    return {'risk_level': r, 'source_cited': s, 'actionable': a, 'lang_correct': l, 'total': r+s+a+l}

def run_eval(mdl, tok, label):
    mdl.eval()
    results = []
    for q in EVAL_QUESTIONS:
        prompt = tok.apply_chat_template(
            [{"role": "user", "content": f"{SYSTEM_INSTRUCTION}\n\n{q['question']}"}],
            tokenize=False, add_generation_prompt=True)
        # Model is forced to cuda:0 — move inputs there
        inputs = tok(prompt, return_tensors="pt")
        inputs = {k: v.to("cuda:0") for k, v in inputs.items()}
        with torch.no_grad():
            out = mdl.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=300,
                temperature=0.3, top_p=0.95,
                do_sample=True, repetition_penalty=1.05,
            )
        ans = tok.decode(out[0][inputs["input_ids"].shape[1]:],
                         skip_special_tokens=True).strip()
        sc = score_response(q, ans)
        results.append({'id': q['id'], 'lang': q['lang'], 'topic': q['topic'],
                        'model': label, 'answer': ans, **sc})
        print(f"[{label}] {q['id']} ({q['lang']}) {sc['total']}/4")
    return results

print(f"Ready: {len(EVAL_QUESTIONS)} questions")

## 4. Load BASE model and evaluate

Plain HuggingFace transformers — no unsloth.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Force model onto cuda:0 only — avoids cross-GPU tensor errors with T4 x2
DEVICE_MAP = {"": 0}

BASE_CANDIDATES = [
    "google/gemma-4-e2b-it",
    "google/gemma-4-2b-it",
]

base_model, raw_proc = None, None
for name in BASE_CANDIDATES:
    try:
        print(f"Trying {name}...")
        raw_proc = AutoProcessor.from_pretrained(name, token=token)
        base_model = AutoModelForCausalLM.from_pretrained(
            name,
            quantization_config=bnb_config,
            device_map=DEVICE_MAP,
            token=token,
            attn_implementation="eager",
        )
        print("Loaded base:", name)
        break
    except Exception as e:
        print("  failed:", e)

if base_model is None:
    raise RuntimeError("No base model loaded.")

text_tok = raw_proc.tokenizer if hasattr(raw_proc, "tokenizer") else raw_proc
text_tok.padding_side = "right"
if text_tok.pad_token is None:
    text_tok.pad_token = text_tok.eos_token

print("Running base eval...")
base_results = run_eval(base_model, text_tok, "base")
print(f"Base done: {len(base_results)} responses.")

## 5. Apply LoRA adapter and evaluate fine-tuned model

PEFT applies the adapter on top of the base model already in memory — no reload needed.

In [ ]:
# Gemma4ClippableLinear wraps Linear4bit — PEFT can't inject into the wrapper.
# Unwrap first to expose the inner Linear4bit, then apply the adapter.

def unwrap_gemma4_clippable(model):
    for parent_name, parent_module in list(model.named_modules()):
        for child_name, child_module in list(parent_module.named_children()):
            if type(child_module).__name__ == "Gemma4ClippableLinear":
                setattr(parent_module, child_name, child_module.linear)
    return model

print("Unwrapping Gemma4ClippableLinear...")
base_model = unwrap_gemma4_clippable(base_model)
print("Unwrap done. Applying LoRA adapter...")

ft_model = PeftModel.from_pretrained(
    base_model,
    "Manjula808/amma-sewana-gemma4-lora",
    token=token,
)
print("Adapter applied. Running fine-tuned eval...")

ft_results = run_eval(ft_model, text_tok, "finetuned")
print(f"Fine-tuned done: {len(ft_results)} responses.")

## 6. Score comparison table

In [ ]:
df = pd.DataFrame(base_results + ft_results)
n  = len(EVAL_QUESTIONS)

print(f"{'='*55}")
print(f"{'Metric':<32} {'Base':>6} {'Fine-tuned':>10}")
print(f"{'='*55}")

b_tot = df[df['model']=='base']['total'].sum()
f_tot = df[df['model']=='finetuned']['total'].sum()
print(f"{'Overall /' + str(n*4):<32} {b_tot:>6} {f_tot:>10}  "
      f"({100*b_tot/(n*4):.0f}% vs {100*f_tot/(n*4):.0f}%)")

for lc, lbl in [('en', 'English /48'), ('si', 'Sinhala /48')]:
    b = df[(df['model']=='base') & (df['lang']==lc)]['total'].sum()
    f = df[(df['model']=='finetuned') & (df['lang']==lc)]['total'].sum()
    print(f"{lbl:<32} {b:>6} {f:>10}")

for c, lbl in [('risk_level', f'Risk level stated /{n}'),
               ('source_cited', f'Source cited /{n}'),
               ('lang_correct', f'Correct language /{n}')]:
    b = df[df['model']=='base'][c].sum()
    f = df[df['model']=='finetuned'][c].sum()
    print(f"{lbl:<32} {b:>6} {f:>10}")
print(f"{'='*55}")

pivot = df.pivot_table(
    index=['id','lang','topic'], columns='model',
    values='total', aggfunc='first'
).reset_index()
pivot['delta'] = pivot['finetuned'] - pivot['base']
print(pivot.sort_values('id').to_string(index=False))

## 7. Side-by-side answers (for writeup)

In [ ]:
bd = {r['id']: r['answer'] for r in base_results}
fd = {r['id']: r['answer'] for r in ft_results}

for qid in ["EN-02", "EN-06", "SI-01", "SI-04"]:
    q = next(x for x in EVAL_QUESTIONS if x['id'] == qid)
    bs = score_response(q, bd[qid])
    fs = score_response(q, fd[qid])
    print("="*65)
    print(f"[{qid}] {q['question']}")
    print(f"\nBASE ({bs['total']}/4):")
    print(textwrap.fill(bd[qid], 65))
    print(f"\nFINE-TUNED ({fs['total']}/4):")
    print(textwrap.fill(fd[qid], 65))
    print()

## 8. Save

In [ ]:
df.to_csv("/kaggle/working/eval_full.csv", index=False)

with open("/kaggle/working/eval_showcase.txt", "w", encoding="utf-8") as fh:
    for qid in ["EN-02", "EN-06", "EN-10", "SI-01", "SI-04", "SI-07"]:
        q = next(x for x in EVAL_QUESTIONS if x['id'] == qid)
        fh.write(f"[{qid}] {q['question']}\n")
        fh.write(f"BASE ({score_response(q, bd[qid])['total']}/4):\n{bd[qid]}\n\n")
        fh.write(f"FINETUNED ({score_response(q, fd[qid])['total']}/4):\n{fd[qid]}\n\n")

print("Saved: eval_full.csv | eval_showcase.txt")
print("Save Version -> Save & Run All (Commit) to lock as artifact.")